# 04 — Classical ML Tier

Trains three classical models on TF-IDF features and reports unified metrics.

Models: LogisticRegression → LinearSVC → LightGBM. Optional imbalanced-learn comparison via RandomUnderSampler is included. SHAP-style coefficients give a quick interpretability figure for the report.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))

import json
import numpy as np
import polars as pl
import joblib
from scipy import sparse
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, roc_auc_score,
)

from utils import ARTIFACTS_DIR, FIGURES_DIR, save_metrics, set_seed
set_seed()
sns.set_theme(style="whitegrid")

In [ ]:
def load_split(name):
    X = sparse.load_npz(ARTIFACTS_DIR / f"tfidf_{name}.npz")
    df = pl.read_parquet(ARTIFACTS_DIR / f"split_{name}.parquet").to_pandas()
    return X, df["label"].values

X_train, y_train = load_split("train")
X_val,   y_val   = load_split("val")
X_test,  y_test  = load_split("test")
print("Train:", X_train.shape, " Test:", X_test.shape)

In [ ]:
def evaluate(name, model, X, y, has_proba=True):
    pred = model.predict(X)
    out = {
        "model": name,
        "accuracy": accuracy_score(y, pred),
        "f1_macro": f1_score(y, pred, average="macro"),
        "f1_pos":   f1_score(y, pred, pos_label=1),
        "f1_neg":   f1_score(y, pred, pos_label=0),
    }
    if has_proba and hasattr(model, "predict_proba"):
        out["roc_auc"] = roc_auc_score(y, model.predict_proba(X)[:, 1])
    return out

results = []

In [ ]:
logreg = LogisticRegression(max_iter=1000, C=1.0, n_jobs=-1, solver="liblinear")
logreg.fit(X_train, y_train)
results.append(evaluate("LogReg", logreg, X_test, y_test))
print(results[-1])

In [ ]:
svc = LinearSVC(C=1.0, max_iter=2000)
svc.fit(X_train, y_train)
# LinearSVC has no predict_proba; use decision_function for AUC
y_score = svc.decision_function(X_test)
res = evaluate("LinearSVC", svc, X_test, y_test, has_proba=False)
res["roc_auc"] = roc_auc_score(y_test, y_score)
results.append(res)
print(results[-1])

In [ ]:
try:
    import lightgbm as lgb
    lgbm = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=64,
        n_jobs=-1,
        random_state=42,
    )
    lgbm.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(20)])
    results.append(evaluate("LightGBM", lgbm, X_test, y_test))
    print(results[-1])
except ImportError:
    print("lightgbm not installed; skipping.")

In [ ]:
# Confusion matrix for the best of the three
import pandas as pd
res_df = pd.DataFrame(results).set_index("model")
best_name = res_df["f1_macro"].idxmax()
print("Best model by macro-F1:", best_name)
print(res_df.round(4))

best = {"LogReg": logreg, "LinearSVC": svc, "LightGBM": locals().get("lgbm")}.get(best_name)
preds = best.predict(X_test)
cm = confusion_matrix(y_test, preds)
fig, ax = plt.subplots(figsize=(4, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["neg", "pos"], yticklabels=["neg", "pos"])
ax.set_title(f"{best_name} confusion matrix")
ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout()
fig_path = FIGURES_DIR / "04_classical_confusion.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Top-coefficient features for LogReg (interpretability)
import pandas as pd
tfidf = joblib.load(ARTIFACTS_DIR / "tfidf_vectorizer.joblib")
vocab = np.array(tfidf.get_feature_names_out())
coef = logreg.coef_.ravel()
top_pos = vocab[np.argsort(coef)[-20:]][::-1]
top_neg = vocab[np.argsort(coef)[:20]]
print("Top POSITIVE features:", list(top_pos))
print("Top NEGATIVE features:", list(top_neg))

pd.DataFrame({"positive_features": top_pos, "negative_features": top_neg}).to_csv(
    ARTIFACTS_DIR / "classical_top_features.csv", index=False
)

In [ ]:
# Imbalanced-learn comparison on a downsampled training set
try:
    from imblearn.under_sampling import RandomUnderSampler
    rus = RandomUnderSampler(random_state=42)
    X_rus, y_rus = rus.fit_resample(X_train, y_train)
    logreg_rus = LogisticRegression(max_iter=1000, n_jobs=-1, solver="liblinear")
    logreg_rus.fit(X_rus, y_rus)
    print("LogReg + RandomUnderSampler:", evaluate("LogReg+RUS", logreg_rus, X_test, y_test))
except ImportError:
    print("imbalanced-learn not installed; skipping.")

In [ ]:
save_metrics("classical_ml", {"results": results, "best_model": best_name})
joblib.dump(best, ARTIFACTS_DIR / f"classical_best_{best_name}.joblib")
print("Saved metrics and best model.")

## Findings to record in Methodology / Results

- Cross-model F1 / accuracy / AUC (table above) — copy directly into the LaTeX Results section.
- Confusion matrix figure: `figures/04_classical_confusion.png`.
- Top positive/negative TF-IDF features (file: `classical_top_features.csv`) — useful qualitative interpretability content for the Discussion.